### Display clustering / kNN results
Notebook to compare two metrics: MTEB, which uses sklearn.cluster.MiniBatchKMeans() and kNN, which uses sklearn.neighbors.KNeighborsClassifier()
MTEB results are saved in .json files, one for each task. 
kNN results are saved in .json files, one for each model.
This notebook aggregates the results for multiple tasks and multiple models.

In [ ]:
import os
import mteb
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import regex as re
import math
from matplotlib.lines import Line2D

In [ ]:
# get task type 
mteb.get_task("ArguAna").metadata.type

In [ ]:
mteb.get_task("ArguAna").metadata.name

#### Analyze MTEB results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/sparse_results/Tfidf"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    #print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df_mteb = pd.DataFrame.from_dict(data, orient='index')
df_mteb.index.name = "task_name"

In [ ]:
task_selection = ["ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P",
                 "RedditClusteringP2P", "StackExchangeClusteringP2P"]
model_exclusion = ["1.0", "svd", "svd_log_piecewise"]

df_mteb_c = df_mteb.loc[task_selection].drop(model_exclusion, axis=1)
df_mteb_c

#### Analyze kNN results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/knn_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
        
    for file in files:
        # Skip unwanted files
        model_name = file.strip(".json")
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                for key, value in json_data.items():
                    if type(value) == list:
                        value = np.mean(value)
                    json_data[key] = round(value*100, 2)
                
                
                data[model_name.removeprefix("Tfidf_").replace("old_", "")] = json_data
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df_knn = pd.DataFrame.from_dict(data)
df_knn

In [ ]:
task_order = ["ArxivClusteringP2P", "arxiv_full", "arxiv_batchwise", "BiorxivClusteringP2P", "biorxiv_full", "biorxiv_batchwise", 
              "MedrxivClusteringP2P", "medrxiv_full", "medrxiv_batchwise", "RedditClusteringP2P", "reddit_full", "reddit_batchwise",
              "StackExchangeClusteringP2P", "stackexchange_full", "stackexchange_batchwise"]
model_order = ["tfidf_log", "svd_log_novocab", "svd_log_old", "svd_log", "svd50_log", "svd200_log", "svd300_log", "svd500_log", "svd768_log", "rnd100_log", "rnd500_log", "rnd768_log"]
df = pd.concat([df_mteb_c, df_knn])
df.index = df.index.str.strip()
df = df.reindex(task_order, columns=model_order)
df

In [ ]:
df.to_latex(na_rep="-", float_format="%.2f")

#### Plotting the results

In [ ]:
# add column specifying the task type
df["task_types"] = ["mteb", "kNN full", "kNN batched"]*5

In [ ]:
df.index.name = "task_name"

In [ ]:
df = df[df["task_types"] != "kNN full"]
df["task_types"] = df["task_types"].replace("kNN batched", "kNN")
df

In [ ]:
df_melted = df.reset_index().melt(id_vars=["task_name", "task_types"], var_name="version", value_name="score")

In [ ]:
df_melted

In [ ]:
# write function to get dimensionality of model
def extract_number(s: str):
    """
    Extracts a number from a string and returns it as an integer.
    If no number is found, returns NaN.
    """
    # catch standard svd, because the size isn't in the name
    if "svd_" in s:
        return 100
    match = re.search(r'\d+', s)
    if match:
        return int(match.group())
    else:
        return math.nan


In [ ]:
# write funciton to get dataset name
def dataset_name(s: str):
    s = s.lower()
    if "arxiv" in s:
        s = "Arxiv"
    elif "bio" in s:
        s = "Biorxiv"
    elif "med" in s:
        s = "Medrxiv"
    elif "reddit" in s:
        s = "Reddit"
    elif "stack" in s:
        s = "StackExchange"
    return s

In [ ]:
# write a function to get reduction type of model
def reduction_type(s: str):
    if "rnd" in s:
        return "random projection"
    elif "old" in s:
        return "svd batchwise"
    elif "vocab" in s:
        return "svd & vocab batchwise"
    elif "svd" in s:
        return "svd"
    else:
        return "no reduction" 

In [ ]:
df_melted["embedding dimensions"] = [extract_number(v) for v in df_melted["version"]]
df_melted["dataset"] = [dataset_name(n) for n in df_melted["task_name"]]
df_melted["reduction type"] = [reduction_type(v) for v in df_melted["version"]]

In [ ]:
df_melted

#### plot version 1
- 5 panels, 1 for each dataset
- x axis: embedding dims
- y axis: scores
- different colors for task types
- different opacities for reduction type? random proj at 50%

In [ ]:
# create color matching for task types
task_types = list(df_melted["task_types"].unique())
#task_colors = dict(zip(task_types, sns.color_palette('colorblind', len(task_types))))
task_colors = dict(zip(task_types, ["darkorange", "royalblue"]))

# create opacity matching for reduction type
reduction_types = ["random projection", "svd"]
opacity_dict = dict(zip(reduction_types, [0.5, 1.0]))

In [ ]:
datasets = list(df_melted["dataset"].unique())

fix, axs = plt.subplots(2,3, figsize=(9,6))

# iterate through axes
k = 0
for i in range(2):
    for j in range(3):
        ax = axs[i,j]
        if i==1 and j ==2:
            ax.axis("off")
        else:
            dataset = datasets[k]
            k += 1
        
            # each subplot shows 1 dataset
            df_dataset = df_melted[df_melted["dataset"] == dataset]

            # draw lineplots for each reduction type
            for red in reduction_types:
                df_subset = df_dataset[df_melted["reduction type"] == red]
                # one line for each task type
                for task in df_subset["task_types"].unique():
                    task_subset = df_subset[df_subset["task_types"] == task]
                    task_subset_sorted = task_subset.sort_values("embedding dimensions")
                    sns.lineplot(data=task_subset_sorted, x="embedding dimensions", y="score",
                                hue="task_types", palette=task_colors, alpha=opacity_dict[red],
                                ax=ax, markers=True, style="dataset")
        
            # design stuff
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

            ax.legend([], [], frameon=False)
            ax.set_ylabel("Performance Score")
            ax.set_ylim(0,78)
        
            ax.set_title(f"{dataset}")


# Create a custom legend
# Legend for task types
task_legend = [
    Line2D([0], [0], color=color, lw=2, label=task)
    for task, color in task_colors.items()
]

# Legend for reduction types (using one example color)
reduction_legend = [
    Line2D([0], [0], color="black", lw=2, alpha=opacity, label=red)
    for red, opacity in opacity_dict.items()
]

# Place the legend in the empty subplot area
handles = task_legend #+ reduction_legend
labels = [line.get_label() for line in handles]
first_legend = axs[1, 2].legend(
    handles,
    labels,
    title="task types",
    frameon=False,
    loc="center left",
    bbox_to_anchor=(0.2, 0.7),  
    title_fontsize='medium',
    fontsize='small',
)

#add legend manually to axes
axs[1, 2].add_artist(first_legend)

handles = reduction_legend
labels = [line.get_label() for line in handles]
axs[1, 2].legend(
    handles,
    labels,
    title="reduction types",
    frameon=False,
    loc="center left",
    bbox_to_anchor=(0.2, 0.3),  
    title_fontsize='medium',
    fontsize='small',
)

# remove axis labels
for i,j in [[0,0],[0,1]]:
    axs[i,j].set(xlabel=None)
for i,j in [[0,1],[0,2],[1,1]]:
    axs[i,j].set(ylabel=None)


plt.tight_layout()
plt.savefig("../img/clustering_dimensions.png")

In [ ]:
datasets

In [ ]:
np.mean([0.63,
        0.586,
        0.598,
        0.577,
        0.608,
        0.602,
        0.566,
        0.564,
        0.612,
        0.578])